In [65]:
from scrape_rugbypass import *
import pandas as pd
from bs4 import BeautifulSoup
import requests
import itertools
import datetime
import os

In [6]:
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://www.rugbypass.com",
    "Referer": "https://www.rugbypass.com/",
})

In [67]:
def retreive_fixtures(years, competition_url):
    payload = {
            "action": "filter-fixtures",
            "team": "0",
            "comp": "0",
            "isContent": "1",
        }
        
    data = {}
        
    for year in years:
        payload["season"] = year
        response = session.post(competition_url, files={k: (None, v) for k, v in payload.items()})
        data[year] = response.json()
    return data

In [106]:
# for URC, Prem and Top14, 2023 refers to 2022/2023
competition_url = "https://www.rugbypass.com/european-champions-cup/fixtures-results/"
years = range(2023, 2027)
data = retreive_fixtures(years, competition_url)

In [49]:
def get_urls(data, year):
    game_days = data[year]["currentGameDays"]
    games = itertools.chain.from_iterable(day["tournaments"][0]["games"] for day in game_days)
    return [{"id": game["id"], "url": game["url"]} for game in games]

In [51]:
six_nations_matches = {year: get_urls(data, year) for year in years}

In [60]:
six_nations_urls = pd.DataFrame([{"year": year, **match} for year in years for match in six_nations_matches[year]])

In [62]:
six_nations_urls.to_parquet("six_nations_urls.parquet")

In [69]:
urc_matches = {year: get_urls(data, year) for year in years}

In [71]:
urc_urls = pd.DataFrame([{"year": year, **match} for year in years for match in urc_matches[year]])

In [86]:
urc_urls.to_parquet("urc_urls.parquet")

In [88]:
top14_matches = {year: get_urls(data, year) for year in years}

In [89]:
top14_urls = pd.DataFrame([{"year": year, **match} for year in years for match in top14_matches[year]])

In [95]:
top14_urls.to_parquet("top14_urls.parquet")

In [99]:
prem_matches = {year: get_urls(data, year) for year in years}
prem_urls = pd.DataFrame([{"year": year, **match} for year in years for match in prem_matches[year]])
prem_urls.to_parquet("prem_urls.parquet")

In [107]:
cc_matches = {year: get_urls(data, year) for year in years}
cc_urls = pd.DataFrame([{"year": year, **match} for year in years for match in cc_matches[year]])
cc_urls.to_parquet("cc_urls.parquet")

In [125]:
competitions = pd.DataFrame({"competition": ["Six Nations", "URC", "Top 14", "Prem", "Champions Cup"]})
competitions.index.name = "competition_id"
competitions

,competition
competition_id,
0,Six Nations
1,URC
2,Top 14
3,Prem
4,Champions Cup


In [143]:
# URC and Top14, Prem are missing around half of their matches from the 2023/2024 and 2024/2025 seasons
six_nations_urls = pd.read_parquet("six_nations_urls.parquet")
urc_urls = pd.read_parquet("urc_urls.parquet")
top14_urls = pd.read_parquet("top14_urls.parquet")
prem_urls = pd.read_parquet("prem_urls.parquet")
cc_urls = pd.read_parquet("cc_urls.parquet")

six_nations_urls["competition_id"] = 0
urc_urls["competition_id"] = 1
top14_urls["competition_id"] = 2
prem_urls["competition_id"] = 3
cc_urls["competition_id"] = 4

urls = pd.concat([six_nations_urls, urc_urls, top14_urls, prem_urls, cc_urls], ignore_index=True)
urls.rename(columns = {"id": "match_id"}, inplace=True)
urls.set_index("match_id", inplace=True)

In [162]:
urls

,year,url,competition_id
match_id,,,
921086,2022,https://www.rugbypass.com/live/ireland-vs-wale...,0
921087,2022,https://www.rugbypass.com/live/england-vs-scot...,0
921088,2022,https://www.rugbypass.com/live/france-vs-italy...,0
921089,2022,https://www.rugbypass.com/live/scotland-vs-wal...,0
921090,2022,https://www.rugbypass.com/live/france-vs-irela...,0
...,...,...,...
948484,2026,https://www.rugbypass.com/live/leinster-vs-sal...,4
948481,2026,https://www.rugbypass.com/live/bordeaux-vs-tou...,4
948486,2026,https://www.rugbypass.com/live/leinster-vs-tou...,4


In [146]:
urls.to_parquet("all_urls.parquet")

In [151]:
[d for d in urls[:10].values]

[array([2022, 'https://www.rugbypass.com/live/ireland-vs-wales/?g=921086',
        0], dtype=object),
 array([2022,
        'https://www.rugbypass.com/live/england-vs-scotland/?g=921087', 0],
       dtype=object),
 array([2022, 'https://www.rugbypass.com/live/france-vs-italy/?g=921088',
        0], dtype=object),
 array([2022, 'https://www.rugbypass.com/live/scotland-vs-wales/?g=921089',
        0], dtype=object),
 array([2022, 'https://www.rugbypass.com/live/france-vs-ireland/?g=921090',
        0], dtype=object),
 array([2022, 'https://www.rugbypass.com/live/england-vs-italy/?g=921091',
        0], dtype=object),
 array([2022,
        'https://www.rugbypass.com/live/france-vs-scotland/?g=921092', 0],
       dtype=object),
 array([2022, 'https://www.rugbypass.com/live/england-vs-wales/?g=921093',
        0], dtype=object),
 array([2022, 'https://www.rugbypass.com/live/ireland-vs-italy/?g=921094',
        0], dtype=object),
 array([2022, 'https://www.rugbypass.com/live/france-vs-wales/